In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DecimalType
from pyspark.sql.functions import col, when, to_date, expr, cast

In [0]:
# Bronze Layer
# Schema setup and Data Load for all the CSV Files

ball_by_ball_schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("over_id", IntegerType(), True),
    StructField("ball_id", IntegerType(), True),
    StructField("innings_no", IntegerType(), True),
    StructField("team_batting", StringType(), True),
    StructField("team_bowling", StringType(), True),
    StructField("striker_batting_position", IntegerType(), True),
    StructField("extra_type", StringType(), True),
    StructField("runs_scored", IntegerType(), True),
    StructField("extra_runs", IntegerType(), True),
    StructField("wides", IntegerType(), True),
    StructField("legbyes", IntegerType(), True),
    StructField("byes", IntegerType(), True),
    StructField("noballs", IntegerType(), True),
    StructField("penalty", IntegerType(), True),
    StructField("bowler_extras", IntegerType(), True),
    StructField("out_type", StringType(), True),
    StructField("caught", IntegerType(), True),
    StructField("bowled", IntegerType(), True),
    StructField("run_out", IntegerType(), True),
    StructField("lbw", IntegerType(), True),
    StructField("retired_hurt", IntegerType(), True),
    StructField("stumped", IntegerType(), True),
    StructField("caught_and_bowled", IntegerType(), True),
    StructField("hit_wicket", IntegerType(), True),
    StructField("obstructing_field", IntegerType(), True),
    StructField("bowler_wicket", IntegerType(), True),
    StructField("match_date", StringType(), True),
    StructField("season", IntegerType(), True),
    StructField("striker", IntegerType(), True),
    StructField("non_striker", IntegerType(), True),
    StructField("bowler", IntegerType(), True),
    StructField("player_out", IntegerType(), True),
    StructField("fielders", IntegerType(), True),
    StructField("striker_match_sk", IntegerType(), True),
    StructField("strikersk", IntegerType(), True),
    StructField("nonstriker_match_sk", IntegerType(), True),
    StructField("nonstriker_sk", IntegerType(), True),
    StructField("fielder_match_sk", IntegerType(), True),
    StructField("fielder_sk", IntegerType(), True),
    StructField("bowler_match_sk", IntegerType(), True),
    StructField("bowler_sk", IntegerType(), True),
    StructField("playerout_match_sk", IntegerType(), True),
    StructField("battingteam_sk", IntegerType(), True),
    StructField("bowlingteam_sk", IntegerType(), True),
    StructField("keeper_catch", IntegerType(), True),
    StructField("player_out_sk", IntegerType(), True),
    StructField("matchdatesk", StringType(), True)
])

ball_by_ball_df = spark.read.format("csv").schema(ball_by_ball_schema)\
    .option("header",True)\
        .load("s3://analytics-etl-s3-bucket/ipl-data-till-2017/Ball_By_Ball.csv")

ball_by_ball_df = ball_by_ball_df.withColumn("match_date",to_date("match_date","M/d/yyyy"))

In [0]:

match_schema = StructType([
    StructField("match_sk", IntegerType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("team1", StringType(), True),
    StructField("team2", StringType(), True),
    StructField("match_date", StringType(), True),
    StructField("season_year", IntegerType(), True),
    StructField("venue_name", StringType(), True),
    StructField("city_name", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("toss_winner", StringType(), True),
    StructField("match_winner", StringType(), True),
    StructField("toss_name", StringType(), True),
    StructField("win_type", StringType(), True),
    StructField("outcome_type", StringType(), True),
    StructField("man_of_match", StringType(), True),
    StructField("win_margin", IntegerType(), True),
    StructField("country_id", IntegerType(), True)
])

match_df = spark.read.format("csv").schema(match_schema)\
    .option("header",True)\
        .load("s3://analytics-etl-s3-bucket/ipl-data-till-2017/Match.csv")

#Date transformation
match_df = match_df.withColumn("match_date", expr("""
                                                    coalesce(
                                                        try_to_date(match_date,"MM/dd/yyyy"),
                                                        try_to_date(match_date,"M/dd/yyyy"),
                                                        try_to_date(match_date,"MM/d/yyyy"),
                                                        try_to_date(match_date,"M/d/yyyy")
                                                        )
                                                    """))

In [0]:
# Schema setup and Data Load for Player Match file

player_match_schema = StructType([
    StructField("player_match_sk", IntegerType(), True),
    StructField("player_match_key", DecimalType(18, 2), True),
    StructField("match_id", IntegerType(), True),
    StructField("player_id", IntegerType(), True),
    StructField("player_name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("batting_hand", StringType(), True),
    StructField("bowling_skill", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("role_desc", StringType(), True),
    StructField("player_team", StringType(), True),
    StructField("opponent_team", StringType(), True),
    StructField("season_year", IntegerType(), True),
    StructField("is_man_of_the_match", IntegerType(), True),
    StructField("age_as_on_match", IntegerType(), True),
    StructField("is_players_team_won", IntegerType(), True),
    StructField("batting_status", StringType(), True),
    StructField("bowling_status", StringType(), True),
    StructField("player_captain", StringType(), True),
    StructField("opponent_captain", StringType(), True),
    StructField("player_keeper", StringType(), True),
    StructField("opponent_keeper", StringType(), True)
])

player_match_df = spark.read.format("csv").schema(player_match_schema)\
    .option("header",True)\
        .load("s3://analytics-etl-s3-bucket/ipl-data-till-2017/Player_match.csv")

#Date Transformation
player_match_df = player_match_df.withColumn("dob", 
                                    expr("""
                                             coalesce(
                                                try_to_date(dob, "MM/dd/yyyy"),
                                                try_to_date(dob, "M/d/yyyy"),
                                                try_to_date(dob, "M/dd/yyyy"),
                                                try_to_date(dob, "MM/d/yyyy")
                                                    )
                                        """
                                        )
                                 )

In [0]:
# Schema setup and Data Load for Player csv file

player_schema = StructType([
    StructField("player_sk", IntegerType(), True),
    StructField("player_id", IntegerType(), True),
    StructField("player_name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("batting_hand", StringType(), True),
    StructField("bowling_skill", StringType(), True),
    StructField("country_name", StringType(), True)
])

player_df = spark.read.format("csv").schema(player_schema)\
    .option("header",True)\
        .load("s3://analytics-etl-s3-bucket/ipl-data-till-2017/Player.csv")

#Date Transformation
player_df = player_df.withColumn("dob",
                                    expr("""
                                             coalesce(
                                                try_to_date(dob, "MM/dd/yyyy"),
                                                try_to_date(dob, "M/d/yyyy"),
                                                try_to_date(dob, "M/dd/yyyy"),
                                                try_to_date(dob, "MM/d/yyyy")
                                                    )
                                        """
                                        )
                                 )

In [0]:
# Schema setup and Data Load for Teams csv file

team_schema = StructType([
    StructField("team_sk", IntegerType(), True),
    StructField("team_id", IntegerType(), True),
    StructField("team_name", StringType(), True)
])

team_df = spark.read.format("csv").schema(team_schema)\
    .option("header",True)\
        .load("s3://analytics-etl-s3-bucket/ipl-data-till-2017/Team.csv")

In [0]:
# Writing data and storing it back as bronze-level in S3 bucket
assert ball_by_ball_df.count() > 0, "ball_by_ball_df is empty!"
print("Writing ball_by_ball to Bronze layer...")
ball_by_ball_df.write.mode("overwrite").parquet("s3://analytics-etl-s3-bucket/bronze-level/ball_by_ball")

assert match_df.count() > 0, "match_df is empty!"
print("Writing Match to Bronze layer...")
match_df.write.mode("overwrite").parquet("s3://analytics-etl-s3-bucket/bronze-level/match")

assert player_match_df.count() > 0, "player_match_df is empty!"
print("Writing PLayer Match to Bronze layer...")
player_match_df.write.mode("overwrite").parquet("s3://analytics-etl-s3-bucket/bronze-level/player_match")

assert player_df.count() > 0, "player_df is empty!"
print("Writing Player to Bronze layer...")
player_df.write.mode("overwrite").parquet("s3://analytics-etl-s3-bucket/bronze-level/player")

assert team_df.count() > 0, "team_df is empty!"
print("Writing Team to Bronze layer...")
team_df.write.mode("overwrite").parquet("s3://analytics-etl-s3-bucket/bronze-level/team")

In [0]:
dbutils.notebook.exit("Bronze Layer is SUCCESS!")